# EMG-ASL Layer: Quick Start Notebook

Run the full pipeline without hardware using synthetic EMG data. All cells run top-to-bottom in under 2 minutes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # project root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.utils.constants import (
    ASL_LABELS, N_CHANNELS, SAMPLE_RATE, WINDOW_SIZE_SAMPLES, FEATURE_VECTOR_SIZE
)
from src.data.synthetic import generate_session
from src.data.loader import create_windows
from src.utils.features import extract_features_batch
from src.models.lstm_classifier import ASLEMGClassifier

print(f"Classes: {len(ASL_LABELS)}")
print(f"Channels: {N_CHANNELS}, Sample rate: {SAMPLE_RATE} Hz")
print(f"Window: {WINDOW_SIZE_SAMPLES} samples = {WINDOW_SIZE_SAMPLES/SAMPLE_RATE*1000:.0f} ms")

## 1. Synthetic EMG Data

In [ ]:
# Generate a synthetic 5-minute session for participant 1
session_df = generate_session(participant_id='P001', seed=42)
print(f"Session shape: {session_df.shape}")
print(f"Duration: {session_df['timestamp_ms'].max() / 1000:.1f} s")
print(f"Labels: {sorted(session_df['label'].unique())[:10]}...")

# Plot first 2 seconds of channel 0
fig, axes = plt.subplots(3, 1, figsize=(12, 6))
t = session_df['timestamp_ms'].values / 1000
axes[0].plot(t[:400], session_df['ch1'].values[:400], linewidth=0.8)
axes[0].set_ylabel('ch1 (mV)')
axes[0].set_title('Synthetic EMG: First 2 Seconds (Channel 1)')
axes[1].plot(t[:400], session_df['ch5'].values[:400], linewidth=0.8, color='orange')
axes[1].set_ylabel('ch5 (mV)')
# Plot label timeline
labels_numeric = [ASL_LABELS.index(l) if l in ASL_LABELS else -1 for l in session_df['label'].values[:400]]
axes[2].step(t[:400], labels_numeric, where='post', linewidth=0.8, color='green')
axes[2].set_ylabel('Label index')
axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 2. Feature Extraction

In [ ]:
windows, labels = create_windows(session_df)
print(f"Windows shape: {windows.shape}")  # (N, 40, 8)
print(f"Labels shape: {labels.shape}")

features = extract_features_batch(windows)
print(f"Features shape: {features.shape}")  # (N, 80)

# Plot feature importance via variance
variances = np.var(features, axis=0)
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(FEATURE_VECTOR_SIZE), variances)
ax.set_xlabel('Feature index')
ax.set_ylabel('Variance across windows')
ax.set_title(f'Feature Variance ({FEATURE_VECTOR_SIZE}-dim vector, 8 channels x 10 features)')
plt.tight_layout()
plt.show()

## 3. Model (LSTM Classifier)

In [ ]:
import torch
from src.models.lstm_classifier import ASLEMGClassifier

model = ASLEMGClassifier(input_size=320, num_classes=len(ASL_LABELS), label_names=ASL_LABELS)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

## 4. End-to-End Pipeline Demo

In [ ]:
from scripts.demo_pipeline import run_demo  # or inline a mini demo
# Just show a single window prediction
window = windows[0:1]  # shape (1, 40, 8)
flat = window.reshape(1, -1)  # (1, 320)
inp = torch.FloatTensor(flat)
with torch.no_grad():
    logits = model(inp)
    probs = torch.softmax(logits, dim=-1)
    top_prob, top_idx = probs.max(dim=-1)
print(f"Predicted: {ASL_LABELS[top_idx.item()]} ({top_prob.item():.1%} confidence)")
print("(Untrained model -- random output. Train with: python scripts/train_synthetic_baseline.py)")

## 5. Next Steps

- Download real datasets: `bash scripts/download_datasets.sh --italian-sl`
- Train on real data: `python scripts/train_real.py --data-dir data/raw/ --epochs 50`
- Run full smoke test: `python scripts/smoke_test.py`
- Start server: `./start-server.sh`